# NC MNIST Sweep: Depth, Activation, Weight Decay

Baseline confirmed: MLP-5, ReLU, wd=1e-4 → NC1 < 0.01 at epoch 310,
feat_norm at T_NC = **1.063**.

This notebook runs the three sweeps that form the paper's core results:

| Sweep | Variable | Configs | Seeds | Est. time |
|---|---|---|---|---|
| H1 | Depth | 2, 3, 7 layers | 3 | ~30 min |
| H2 | Activation | GELU, Tanh | 3 | ~20 min |
| H3 | Weight decay | 1e-5, 5e-5, 1e-4, 5e-4 | 3 | ~40 min |

**Total: ~90 min on T4.**

The key question answered by panel (c): is feat_norm at T_NC
consistent across all conditions (CV < 15%)?

**Fully self-contained. Settings → T4 GPU → Run All.**

In [1]:
import torch, torchvision, time
import torchvision.transforms as T
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE_DIR = '/kaggle/working/'
torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True
print(f'Device: {DEVICE} | GPU: {torch.cuda.get_device_name(0)}')

# Baseline result to compare against
BASELINE_FN = 1.0628   # feat_norm at T_NC from two-phase baseline
BASELINE_TNC = 310     # T_NC epoch from baseline
print(f'Baseline: feat_norm at T_NC = {BASELINE_FN}  T_NC = {BASELINE_TNC}')


Device: cuda | GPU: Tesla T4
Baseline: feat_norm at T_NC = 1.0628  T_NC = 310


In [2]:
transform = T.Compose([T.ToTensor(), T.Normalize((0.1307,),(0.3081,))])
trainset  = torchvision.datasets.MNIST('/kaggle/working/data',
    train=True,  download=True, transform=transform)
testset   = torchvision.datasets.MNIST('/kaggle/working/data',
    train=False, download=True, transform=transform)
train_loader = DataLoader(trainset, batch_size=256, shuffle=True,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(testset,  batch_size=512, shuffle=False,
                          num_workers=2, pin_memory=True)
print(f'MNIST ready: {len(trainset):,} train')


  0%|          | 0.00/9.91M [00:00<?, ?B/s]

 15%|█▍        | 1.47M/9.91M [00:00<00:00, 13.4MB/s]

100%|██████████| 9.91M/9.91M [00:00<00:00, 59.0MB/s]

  0%|          | 0.00/28.9k [00:00<?, ?B/s]

100%|██████████| 28.9k/28.9k [00:00<00:00, 1.65MB/s]

...

 76%|███████▌  | 1.25M/1.65M [00:00<00:00, 11.2MB/s]

100%|██████████| 1.65M/1.65M [00:00<00:00, 14.6MB/s]

  0%|          | 0.00/4.54k [00:00<?, ?B/s]

100%|██████████| 4.54k/4.54k [00:00<00:00, 9.69MB/s]

MNIST ready: 60,000 train


In [3]:
class MLP5(nn.Module):
    def __init__(self, depth=5, width=512, act_cls=nn.ReLU, num_classes=10):
        super().__init__()
        # depth = number of nonlinear hidden layers
        layers = [nn.Flatten(), nn.Linear(784, width), act_cls()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), act_cls()]
        self.body   = nn.Sequential(*layers)
        self.head   = nn.Linear(width, num_classes)
        self._feats = None
        self.body.register_forward_hook(
            lambda m, i, o: setattr(self, '_feats', o.detach()))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)
    def forward(self, x):
        return self.head(self.body(x))
    def get_features(self, x):
        self(x); return self._feats
    def get_classifier_weights(self):
        return self.head.weight.detach()

print('MLP5 defined (depth parameter added).')


MLP5 defined (depth parameter added).


In [4]:
@torch.no_grad()
def compute_nc(model, loader, K=10):
    model.eval()
    fl, ll = [], []
    for x, y in loader:
        fl.append(model.get_features(x.to(DEVICE)).cpu())
        ll.append(y)
    H = torch.cat(fl).float(); Y = torch.cat(ll)
    mu_G = H.mean(0)
    mu_c = torch.stack([H[Y==c].mean(0) for c in range(K)])
    M    = mu_c - mu_G
    Sw   = sum((H[Y==c]-mu_c[c]).T@(H[Y==c]-mu_c[c]) for c in range(K))/len(H)
    Sb   = M.T @ M / K
    nc1  = (torch.trace(Sw)/torch.trace(Sb).clamp(1e-10)).item()
    Mn   = F.normalize(M, dim=1)
    cos  = Mn @ Mn.T
    mask = ~torch.eye(K, dtype=torch.bool)
    nc2  = (cos[mask]-(-1.0/(K-1))).abs().mean().item()
    Wn   = F.normalize(model.get_classifier_weights().cpu(), dim=1)
    nc3  = (1-(Mn*Wn).sum(1).mean()).item()
    return {'nc1':nc1,'nc2':nc2,'nc3':nc3,
            'feat_norm':H.norm(dim=1).mean().item()}

def evaluate(model, loader):
    model.eval(); correct=total=0
    with torch.no_grad():
        for x,y in loader:
            x,y = x.to(DEVICE),y.to(DEVICE)
            correct += (model(x).argmax(1)==y).sum().item()
            total   += len(y)
    return correct/total

def run_twophase(model, name, lr=1e-3, wd=1e-4,
                 phase1=200, phase2=400, nc_every=10):
    model = model.to(DEVICE)
    K = 10; rows = []; terminal = False; t0 = time.time()
    for phase, loss_fn, n_ep in [
        (1, 'ce',  phase1),
        (2, 'mse', phase2),
    ]:
        opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_ep)
        offset = phase1 if phase == 2 else 0
        for ep_l in range(1, n_ep+1):
            ep = offset + ep_l
            model.train()
            for x, y in train_loader:
                x,y = x.to(DEVICE,non_blocking=True),y.to(DEVICE,non_blocking=True)
                opt.zero_grad(set_to_none=True)
                logits = model(x)
                loss = F.mse_loss(logits,F.one_hot(y,K).float()) \
                       if loss_fn=='mse' else F.cross_entropy(logits,y)
                loss.backward(); opt.step()
            sched.step()
            if ep_l % nc_every == 0 or ep_l == n_ep:
                tr = evaluate(model, train_loader)
                te = evaluate(model, test_loader)
                if tr >= 0.99 and not terminal:
                    terminal = True
                    print(f'  [{name}] Terminal ep={ep}')
                nc = compute_nc(model, train_loader) if terminal else \
                     {'nc1':None,'nc2':None,'nc3':None,'feat_norm':None}
                rows.append({'epoch':ep,'phase':phase,'train':tr,'test':te,**nc})
                nc1s = f"{nc['nc1']:.5f}" if nc['nc1'] else 'N/A'
                fns  = f"{nc['feat_norm']:.3f}" if nc['feat_norm'] else 'N/A'
                print(f'  ep={ep:>4} tr={tr:.4f} nc1={nc1s} fn={fns} '
                      f't={(time.time()-t0)/60:.1f}m')
                if nc['nc1'] is not None and nc['nc1'] < 0.01:
                    print(f'  *** T_NC={ep} fn={nc["feat_norm"]:.4f}')
                    return pd.DataFrame(rows), ep, nc['feat_norm']
    return pd.DataFrame(rows), None, None

print('run_twophase ready.')


run_twophase ready.


In [5]:
# H1: Depth sweep (2, 3, 5, 7 hidden layers) — 3 seeds each
# Baseline depth=5 already done (T_NC=310, fn=1.063)
# Question: do shallower/deeper networks collapse at same feat_norm?
print('='*55)
print('H1: DEPTH SWEEP')
print('='*55)

depth_results = []
for depth in [2, 3, 7]:
    for seed in range(3):
        name = f'depth{depth}-s{seed}'
        print(f'\n--- depth={depth} seed={seed} ---')
        torch.manual_seed(seed)
        model = MLP5(depth=depth, width=512, act_cls=nn.ReLU)
        df, t_nc, fn = run_twophase(model, name, lr=1e-3, wd=1e-4)
        df.to_csv(f'{SAVE_DIR}depth{depth}_s{seed}.csv', index=False)
        depth_results.append({
            'depth': depth, 'seed': seed,
            'T_NC': t_nc, 'feat_norm': fn,
            'test_acc': df.test.iloc[-1]
        })
        print(f'  => T_NC={t_nc}  feat_norm={fn}')

df_depth = pd.DataFrame(depth_results)
df_depth.to_csv(SAVE_DIR + 'sweep_depth.csv', index=False)
print('\nDepth sweep summary:')
print(df_depth.groupby('depth')[['T_NC','feat_norm']].agg(['mean','std']).to_string())


H1: DEPTH SWEEP

--- depth=2 seed=0 ---


  [depth2-s0] Terminal ep=10


  ep=  10 tr=0.9929 nc1=1.07869 fn=31.514 t=1.3m


  ep=  20 tr=0.9952 nc1=0.95316 fn=28.359 t=2.6m


  ep=  30 tr=0.9963 nc1=0.90799 fn=25.529 t=3.9m


...

  ep= 290 tr=0.9954 nc1=0.01134 fn=1.062 t=38.6m


  ep= 300 tr=0.9951 nc1=0.01223 fn=1.068 t=39.9m


  ep= 310 tr=0.9946 nc1=0.01285 fn=1.072 t=41.2m


  ep= 320 tr=0.9946 nc1=0.01236 fn=1.067 t=42.5m


  ep= 330 tr=0.9973 nc1=0.00827 fn=1.069 t=43.9m
  *** T_NC=330 fn=1.0692
  => T_NC=330  feat_norm=1.0691981315612793

Depth sweep summary:
        T_NC       feat_norm          
        mean   std      mean       std
depth                                 
2        NaN   NaN       NaN       NaN
3        NaN   NaN       NaN       NaN
7      340.0  10.0  1.111598  0.040028


In [6]:
# H2: Activation sweep (ReLU, GELU, Tanh) — depth=5, 3 seeds each
# Baseline ReLU already done (T_NC=310, fn=1.063)
# Question: does feat_norm at T_NC vary with activation?
print('='*55)
print('H2: ACTIVATION SWEEP')
print('='*55)

act_results = []
for act_name, act_cls in [('GELU', nn.GELU), ('Tanh', nn.Tanh)]:
    for seed in range(3):
        name = f'{act_name}-s{seed}'
        print(f'\n--- {act_name} seed={seed} ---')
        torch.manual_seed(seed)
        model = MLP5(depth=5, width=512, act_cls=act_cls)
        df, t_nc, fn = run_twophase(model, name, lr=1e-3, wd=1e-4)
        df.to_csv(f'{SAVE_DIR}act{act_name}_s{seed}.csv', index=False)
        act_results.append({
            'act': act_name, 'seed': seed,
            'T_NC': t_nc, 'feat_norm': fn,
            'test_acc': df.test.iloc[-1]
        })
        print(f'  => T_NC={t_nc}  feat_norm={fn}')

df_act = pd.DataFrame(act_results)
df_act.to_csv(SAVE_DIR + 'sweep_act.csv', index=False)
print('\nActivation sweep summary:')
print(df_act.groupby('act')[['T_NC','feat_norm']].agg(['mean','std']).to_string())


H2: ACTIVATION SWEEP

--- GELU seed=0 ---


  [GELU-s0] Terminal ep=10


  ep=  10 tr=0.9951 nc1=0.17514 fn=21.888 t=1.3m


  ep=  20 tr=0.9945 nc1=0.11145 fn=19.373 t=2.6m


  ep=  30 tr=0.9982 nc1=0.09418 fn=19.078 t=3.9m


...

  ep= 270 tr=0.9852 nc1=0.04447 fn=1.709 t=34.3m


  ep= 280 tr=0.9937 nc1=0.03374 fn=1.696 t=35.5m


  ep= 290 tr=0.9939 nc1=0.03365 fn=1.730 t=36.8m


  ep= 300 tr=0.9932 nc1=0.03471 fn=1.694 t=38.1m


  ep= 310 tr=0.9919 nc1=0.03831 fn=1.690 t=39.3m


In [ ]:
# H3: Weight decay sweep — depth=5, ReLU, 3 seeds each
# Question: does feat_norm at T_NC stay consistent across lambda?
# (This is the key threshold hypothesis test)
print('='*55)
print('H3: WEIGHT DECAY SWEEP (key threshold test)')
print('='*55)

wd_results = []
for wd in [1e-5, 5e-5, 1e-4, 5e-4]:
    for seed in range(3):
        name = f'wd{wd}-s{seed}'
        print(f'\n--- wd={wd} seed={seed} ---')
        torch.manual_seed(seed)
        model = MLP5(depth=5, width=512, act_cls=nn.ReLU)
        df, t_nc, fn = run_twophase(model, name, lr=1e-3, wd=wd)
        df.to_csv(f'{SAVE_DIR}wd{str(wd).replace(".","p")}_s{seed}.csv',
                  index=False)
        wd_results.append({
            'wd': wd, 'seed': seed,
            'T_NC': t_nc, 'feat_norm': fn,
            'test_acc': df.test.iloc[-1]
        })
        print(f'  => T_NC={t_nc}  feat_norm={fn}')

df_wd = pd.DataFrame(wd_results)
df_wd.to_csv(SAVE_DIR + 'sweep_wd.csv', index=False)
print('\nWeight decay sweep summary:')
print(df_wd.groupby('wd')[['T_NC','feat_norm']].agg(['mean','std']).to_string())


In [ ]:
# Load all sweep results
import os
dfs = {}
for fname, key in [
    ('sweep_depth.csv', 'depth'),
    ('sweep_act.csv',   'act'),
    ('sweep_wd.csv',    'wd'),
]:
    p = SAVE_DIR + fname
    if os.path.exists(p): dfs[key] = pd.read_csv(p)

plt.rcParams.update({'font.family':'serif','font.size':11,
    'axes.spines.top':False,'axes.spines.right':False})
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Collect ALL feat_norm at T_NC values
all_fns = [BASELINE_FN]

# (a) T_NC by depth
ax = axes[0]
if 'depth' in dfs:
    d = dfs['depth']
    all_fns += d.feat_norm.dropna().tolist()
    g = d.groupby('depth')['T_NC'].agg(['mean','std'])
    ax.errorbar(g.index, g['mean'], yerr=g['std'],
                fmt='o-', color='#2196F3', lw=2, ms=8, capsize=5)
    ax.axhline(BASELINE_TNC, color='gray', ls='--', lw=1,
               label=f'Baseline (depth=5) T_NC={BASELINE_TNC}')
ax.set(xlabel='Depth (hidden layers)', ylabel='T_NC (epoch)',
       title='(a) Depth vs T_NC')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# (b) T_NC by activation
ax = axes[1]
if 'act' in dfs:
    d = dfs['act']
    all_fns += d.feat_norm.dropna().tolist()
    acts = d.groupby('act')['T_NC'].agg(['mean','std']).reset_index()
    # Add baseline ReLU
    baseline_row = pd.DataFrame([{'act':'ReLU','mean':BASELINE_TNC,'std':0}])
    acts = pd.concat([acts, baseline_row], ignore_index=True)
    acts_sorted = acts.sort_values('mean')
    colors_act = {'GELU':'#4CAF50','ReLU':'#F44336','Tanh':'#FF9800'}
    ax.bar(acts_sorted.act,
           acts_sorted['mean'],
           yerr=acts_sorted['std'].fillna(0),
           color=[colors_act.get(a,'#9E9E9E') for a in acts_sorted.act],
           capsize=5, alpha=0.85, edgecolor='black', lw=0.5)
ax.set(xlabel='Activation', ylabel='T_NC (epoch)',
       title='(b) Activation vs T_NC')
ax.grid(axis='y', alpha=0.3)

# (c) THE KEY RESULT: feat_norm at T_NC across ALL conditions
ax = axes[2]
if 'wd' in dfs:
    all_fns += dfs['wd'].feat_norm.dropna().tolist()
if len(all_fns) > 1:
    mean_fn = np.mean(all_fns)
    std_fn  = np.std(all_fns)
    cv      = std_fn / mean_fn
    ax.scatter(range(len(all_fns)), sorted(all_fns),
               color='#E91E63', s=60, zorder=3)
    ax.axhline(mean_fn, color='black', ls='--', lw=1.5,
               label=f'Mean={mean_fn:.3f}')
    ax.fill_between(range(len(all_fns)),
                    mean_fn-std_fn, mean_fn+std_fn,
                    alpha=0.15, color='black')
    ax.set(xlabel='Configuration (sorted)', ylabel='Feature norm at T_NC',
           title=f'(c) Feature norm threshold\nMean={mean_fn:.3f}  '
                 f'CV={cv:.3f}')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
    print(f'THRESHOLD RESULT: mean={mean_fn:.4f}  std={std_fn:.4f}  CV={cv:.3f}')
    if cv < 0.15:
        print('CV < 15% => CONSISTENT threshold — paper central result!')
    else:
        print(f'CV={cv:.3f} — threshold varies with config')

fig.suptitle('NC Sweep: Depth, Activation, Weight Decay | MLP-5 | MNIST',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(SAVE_DIR + 'fig_nc_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_nc_sweep.png')
